# Mask R-CNN ResNet-50 FPN v2 (COCO) — DIMER instance segmentation and bounded fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/tutorials/maskrcnn_instance_segmentation_colab.ipynb) [![torchvision](https://img.shields.io/badge/torchvision-maskrcnn__resnet50__fpn__v2-ee4c2c?style=flat&logo=pytorch&logoColor=white)](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html) [![Upstream](https://img.shields.io/badge/Upstream-pytorch%2Fvision-181717?style=flat&logo=github&logoColor=white)](https://github.com/pytorch/vision) [![arXiv](https://img.shields.io/badge/arXiv-1703.06870-b31b1b.svg)](https://arxiv.org/abs/1703.06870) [![License](https://img.shields.io/badge/License-BSD--3--Clause-green.svg)](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.1 — **standalone** (§4)  
**Capability:** instance segmentation over the COCO classes with torchvision's Mask R-CNN ResNet-50 FPN v2, and a bounded fine-tune that re-heads the box and mask predictors onto your own class vocabulary, evaluates the result on a held-out split with box and mask average precision, and exports a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/maskrcnn_instance_segmentation_pipeline/`, at revision `facebca0a134`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and `download.pytorch.org` for the checkpoint file pinned by SHA-256 `unpinned` (one file that torchvision's weight metadata lists as 177.219 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned checkpoint, runs COCO instance segmentation on a drawn scene, validates the 40-image sign dataset, splits it into training and held-out parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, segments unseen images, exports the adapter, reloads it onto a fresh base model to verify the instances, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.1 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the same validation, segmentation and evaluation-report stages as the sample scene. `USE_BYOD_DATASET` takes your own labelled instance masks through the full adaptation workflow — validate, split, baseline, fine-tune, evaluate, export and reload — under NOTEBOOK_SPEC 2.1 DAT14. Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_DIR` to read from a location without an upload dialog (EXE2).

Mask R-CNN is a two-stage instance segmentation model. A ResNet-50 backbone with a feature pyramid network (FPN) produces feature maps at five scales; a region proposal network (RPN) proposes candidate boxes; and for each surviving region the ROI heads predict a class, a refined box, and a 28×28 mask that is pasted back into the image. This notebook uses torchvision's **v2** weights (`MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1`), trained with the improved recipe from the ViTDet benchmarking paper; torchvision reports 47.4 box AP and 41.8 mask AP on COCO val2017 for them. The model's own transform resizes each image so its shorter side is 800 px, and every instance whose class probability reaches a caller-owned threshold is returned as an xyxy box plus a boolean mask in input pixels.

**The default path really adapts the model:** it re-heads Mask R-CNN onto a three-class traffic sign vocabulary that does not exist in COCO (`stop-sign`, `yield-sign`, `speed-limit-sign`), measures a pre-adaptation baseline, runs a bounded fine-tune with the ResNet-50 body frozen, scores the result on a held-out split with box and mask average precision (AP@[.50:.95] and AP50), runs the adapted model on unseen images, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same instances. Every number you see is measured in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the pinned checkpoint; run COCO instance segmentation on a drawn scene and score per-object `box_iou` and `mask_iou`; probe the model with a blank and a noise image; build and validate a labelled instance-segmentation dataset over a new three-class sign vocabulary; split it and measure a pre-adaptation baseline; run a bounded fine-tune with Mask R-CNN's own training losses; score the adapted model on the held-out split with box and mask AP; run inference on unseen images; and export, reload and verify the adapter.

**This notebook does not demonstrate:** real-world traffic sign segmentation (the adaptation dataset is drawn in code, so the model learns these renderings and nothing about road photographs); COCO benchmark results (the average-precision helper here is a compact implementation without pycocotools area ranges or crowd handling, and it is run on synthetic data only); full-schedule Mask R-CNN training (the v2 weights come from a long multi-GPU recipe, recorded in torchvision's weight metadata as pytorch/vision pull request 5773; the tutorial runs a few epochs on 30 images); semantic or panoptic segmentation; keypoints; video tracking.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA GPU such as a Colab or Kaggle T4 is the documented runtime for the fine-tuning stages and is used automatically when present; the notebook also runs on CPU, much more slowly. Runtimes are not measured in this revision. The pinned `torch==2.14.0` wheel and the checkpoint are the largest downloads.
- **Knowledge:** basic Python, PIL and NumPy; bounding boxes as xyxy pixel coordinates; binary masks as boolean arrays; intersection-over-union (IoU) for boxes and for masks; and how to read average precision (AP50 and AP@[.50:.95]).
- **Data:** the default path generates everything in code with `samples.py` and downloads no dataset: one 640×480 COCO demonstration scene and a 40-image labelled sign dataset, each object with an exact mask. BYOD is optional and off by default. Expected BYOD input: one image, or a directory holding `annotations.json` — a list of `{'file': 'name.png', 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...], 'masks': ['name_mask0.png', ...]}` objects — and the image and mask files it names (one single-channel PNG per instance, non-zero pixels inside the instance). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** `download.pytorch.org` only, to fetch the pinned `torchvision/maskrcnn_resnet50_fpn_v2` checkpoint (one file that torchvision's weight metadata lists as 177.219 MB) at revision `unpinned…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchvision`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'maskrcnn-instance-segmentation-pipeline',
    'repository_revision': 'facebca0a134cff129369deda4c93cb4abe38799',
    'embedded_module': 'src/maskrcnn_instance_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/maskrcnn_instance_segmentation_pipeline/pipeline.py', 'src/maskrcnn_instance_segmentation_pipeline/samples.py'],
    'module_sha256': '01c35cb56c2692e57d047292247e1d7f4ed5f242d14e5246997e698c99ac5095',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/maskrcnn_instance_segmentation_pipeline/` @ `facebca0a134`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/maskrcnn_instance_segmentation_pipeline/pipeline.py`

In [ ]:
"""COCO instance segmentation and bounded fine-tuning with torchvision's Mask R-CNN ResNet-50 FPN v2 weights.

The model is torchvision's own ``maskrcnn_resnet50_fpn_v2`` architecture, built with ``weights=None`` so
that nothing is downloaded at construction, then loaded from one digest-verified local checkpoint file
(``weights/<key>/``) with ``torch.load(..., weights_only=True)``, which refuses arbitrary pickled objects.

Until ``tools/pin_snapshot.py`` has recorded the checkpoint's SHA-256 and byte size, the package refuses to
stage, verify or load weights: an unpinned checkpoint is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "torchvision/maskrcnn_resnet50_fpn_v2"
# The pinned identity of a URL-hosted checkpoint is the SHA-256 of its bytes. torchvision names the file
# after the first 8 hex digits of that digest, so the URL is content-addressed; the full digest is pinned.
MODEL_REVISION = "unpinned"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "maskrcnn-resnet50-fpn-v2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "maskrcnn-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"
WEIGHTS_FILE = "maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth"
WEIGHTS_URL = f"https://download.pytorch.org/models/{WEIGHTS_FILE}"
WEIGHTS_SHA256_PREFIX = "73cbd019"
TORCHVISION_WEIGHTS = "MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1"

# torchvision's COCO category list for these weights (MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1.meta
# ["categories"]), in label-id order: 91 slots. Id 0 is the background class and the 10 "N/A" ids are COCO
# category numbers that COCO 2017 never annotated, which leaves 80 object classes. LABELS drops the
# background, so model label id k is LABELS[k - 1].
COCO_CATEGORIES = (
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "N/A", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse",
    "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "N/A", "backpack", "umbrella", "N/A", "N/A", "handbag",
    "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
    "skateboard", "surfboard", "tennis racket", "bottle", "N/A", "wine glass", "cup", "fork", "knife", "spoon",
    "bowl", "banana", "apple", "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake",
    "chair", "couch", "potted plant", "bed", "N/A", "dining table", "N/A", "N/A", "toilet", "N/A", "tv", "laptop",
    "mouse", "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "N/A",
    "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
)  # fmt: skip
LABELS = COCO_CATEGORIES[1:]

# Score threshold: the value torchvision's visualization-utilities gallery example uses to keep Mask R-CNN
# instances (score_threshold = 0.8), with its mask probability cut (proba_threshold = 0.5). Scores are
# softmax class probabilities after torchvision's per-class NMS; neither value was calibrated for any
# deployment, and the deployment owns tuning them on labelled images.
DETECTION_THRESHOLD = 0.8
MASK_THRESHOLD = 0.5
EVAL_DETECTION_THRESHOLD = 0.05
MAX_DETECTIONS = 100  # torchvision's box_detections_per_img default
MAX_EVAL_DETECTIONS = 100
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_CLASSES = 1000
MAX_RECORDS = 5000

# Training defaults for the bounded tutorial adaptation (torchvision's detection fine-tuning tutorial
# uses SGD, learning rate 0.005, momentum 0.9 and weight decay 0.0005).
DEFAULT_EPOCHS = 10
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 0.005
DEFAULT_MOMENTUM = 0.9
DEFAULT_WEIGHT_DECAY = 0.0005
DEFAULT_SEED = 20260924
BACKBONE_PREFIX = "backbone.body."

COCO_IOU_THRESHOLDS: tuple[float, ...] = tuple(round(0.50 + 0.05 * i, 2) for i in range(10))


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION is the checkpoint's 64-hex SHA-256."""
    revision = MODEL_REVISION
    return len(revision) == 64 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned checkpoint digest yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the SHA-256 and byte size"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local checkpoint against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256") or entry.get("bytes") is None:
            raise ValueError(f"{entry['path']}: manifest records no sha256/bytes; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"checkpoint file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _url_download(entry: Mapping[str, Any], root: Path) -> None:
    """Fetch one manifest entry from its URL; torch.hub checks the filename's SHA-256 prefix on the way in."""
    from torch.hub import download_url_to_file

    download_url_to_file(entry["url"], str(root / entry["path"]), hash_prefix=WEIGHTS_SHA256_PREFIX, progress=False)


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[Mapping[str, Any], Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"checkpoint at {root} is missing {[e['path'] for e in missing]}; pass allow_download=True to fetch it"
        )
    fetch = downloader or _url_download
    for entry in missing:
        if entry.get("url") != WEIGHTS_URL:
            raise ValueError(f"manifest URL {entry.get('url')!r} != package URL {WEIGHTS_URL!r}")
        fetch(entry, root)
    return [entry["path"] for entry in missing]


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of the same shape."""
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    if a.shape != b.shape:
        raise ValueError(f"mask shapes differ: {a.shape} vs {b.shape}")
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    kind: str = "box",
    iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
) -> dict[str, Any]:
    """Compact COCO-style average precision over a scored dataset, on boxes or on masks.

    For each class and IoU threshold, detections are matched greedily to references by descending score;
    each detection matches at most one reference, and precision is sampled at 101 recall points. Classes
    with no reference are left out of the mean. There are no area ranges and no crowd handling.
    """
    if kind not in ("box", "mask"):
        raise ValueError(f"kind must be 'box' or 'mask', got {kind!r}")
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists but {len(references)} references")
    recall_points = np.linspace(0.0, 1.0, 101)
    per_threshold: dict[float, dict[str, float]] = {}

    def overlap(det: Mapping[str, Any], reference: Mapping[str, Any], j: int) -> float:
        if kind == "box":
            return box_iou(det["box"], reference["boxes"][j])
        return mask_iou(det["mask"], reference["masks"][j])

    for threshold in iou_thresholds:
        per_class: dict[str, float] = {}
        for name in class_names:
            scored: list[tuple[float, bool]] = []
            n_references = 0
            for dets, reference in zip(predictions, references, strict=True):
                ref_ids = [j for j, label in enumerate(reference["labels"]) if label == name]
                n_references += len(ref_ids)
                claimed = dict.fromkeys(ref_ids, False)
                for det in sorted((d for d in dets if d["label"] == name), key=lambda d: -float(d["score"])):
                    best, best_iou = -1, 0.0
                    for j in ref_ids:
                        if claimed[j]:
                            continue
                        value = overlap(det, reference, j)
                        if value > best_iou:
                            best, best_iou = j, value
                    hit = best >= 0 and best_iou >= threshold
                    if hit:
                        claimed[best] = True
                    scored.append((float(det["score"]), hit))
            if n_references == 0:
                continue
            if not scored:
                per_class[name] = 0.0
                continue
            scored.sort(key=lambda pair: -pair[0])
            true_positives = np.cumsum([1 if hit else 0 for _s, hit in scored])
            false_positives = np.cumsum([0 if hit else 1 for _s, hit in scored])
            recall = true_positives / n_references
            precision = np.maximum.accumulate((true_positives / np.maximum(true_positives + false_positives, 1))[::-1])[::-1]
            sampled = np.zeros_like(recall_points)
            indices = np.searchsorted(recall, recall_points, side="left")
            valid = indices < len(precision)
            sampled[valid] = precision[indices[valid]]
            per_class[name] = float(sampled.mean())
        per_threshold[threshold] = per_class
    means = {t: (float(np.mean(list(v.values()))) if v else 0.0) for t, v in per_threshold.items()}
    return {
        "ap": float(np.mean(list(means.values()))) if means else 0.0,
        "ap50": means.get(0.5, 0.0),
        "ap75": means.get(0.75, 0.0),
        "per_class_ap50": per_threshold.get(0.5, {}),
        "iou_thresholds": [float(t) for t in iou_thresholds],
        "kind": kind,
        "n_images": len(references),
        "n_references": sum(len(r["labels"]) for r in references),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any, name: str = "threshold") -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


def _check_class_names(class_names: Sequence[str]) -> tuple[str, ...]:
    names = tuple(class_names)
    if not 1 <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class_names must hold 1..{MAX_CLASSES} names, got {len(names)}")
    for name in names:
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"class names must be non-empty strings, got {name!r}")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"class_names contains duplicates: {duplicates}")
    return names


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "mask_threshold": MASK_THRESHOLD,
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB and to a float tensor in [0, 1]; the model's own transform resizes it so the "
        "shorter side is 800 px and the longer at most 1333 px and normalises it with the ImageNet mean and "
        "standard deviation; boxes and masks are returned in input pixels"
    ),
}

DATASET_SCHEMA: dict[str, Any] = {
    "record": (
        "{'image': PIL.Image.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...], "
        "'masks': [bool array (height, width), ...]}"
    ),
    "boxes": "xyxy pixel coordinates inside the image, x0 < x1 and y0 < y1, one per instance",
    "masks": "one non-empty boolean mask per instance, the size of the image, lying inside its box",
    "labels": "one name per instance, each one of class_names",
    "records": [1, MAX_RECORDS],
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request."""
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = DEFAULT_EPOCHS,
) -> dict[str, Any]:
    """Validation stage for labelled instance-segmentation records: raise on the first broken record, else
    return the dataset manifest. Classes that never occur are reported as findings, not silently accepted."""
    names = _check_class_names(class_names)
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    valid = set(names)
    per_class = dict.fromkeys(names, 0)
    total = 0
    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image", "boxes", "labels", "masks"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image', 'boxes', 'labels' and 'masks'")
        try:
            img = validate_image(record["image"])
        except (TypeError, ValueError) as exc:
            raise type(exc)(f"record {idx}: {exc}") from exc
        boxes, labels, masks = record["boxes"], record["labels"], record["masks"]
        if not len(boxes) == len(labels) == len(masks):
            raise ValueError(f"record {idx}: {len(boxes)} boxes, {len(labels)} labels and {len(masks)} masks")
        width, height = img.size
        for b_idx, (box, mask) in enumerate(zip(boxes, masks, strict=True)):
            if len(box) != 4:
                raise ValueError(f"record {idx} box {b_idx} must have 4 elements, got {len(box)}")
            x0, y0, x1, y1 = (float(v) for v in box)
            if not all(np.isfinite([x0, y0, x1, y1])):
                raise ValueError(f"record {idx} box {b_idx} has a non-finite coordinate")
            if not (0.0 <= x0 < x1 <= width and 0.0 <= y0 < y1 <= height):
                raise ValueError(f"record {idx} box {b_idx} [{x0}, {y0}, {x1}, {y1}] is empty or outside image bounds {(width, height)}")
            array = np.asarray(mask)
            if array.shape != (height, width):
                raise ValueError(f"record {idx} mask {b_idx} has shape {array.shape}, image is {(height, width)}")
            if array.dtype != bool and not np.isin(array, (0, 1)).all():
                raise ValueError(f"record {idx} mask {b_idx} must be boolean or 0/1")
            ys, xs = np.nonzero(array)
            if not len(xs):
                raise ValueError(f"record {idx} mask {b_idx} is empty")
            if xs.min() < x0 - 1 or xs.max() + 1 > x1 + 1 or ys.min() < y0 - 1 or ys.max() + 1 > y1 + 1:
                raise ValueError(f"record {idx} mask {b_idx} extends outside its box")
        for label in labels:
            if label not in valid:
                raise ValueError(f"record {idx} has unknown class {label!r}; expected one of {list(names)}")
            per_class[label] += 1
        total += len(labels)
    return {
        "schema": dict(DATASET_SCHEMA),
        "n_records": len(records),
        "n_instances": total,
        "class_names": list(names),
        "instances_per_class": per_class,
        "epochs": epochs,
        "findings": [f"class {name!r} has no instance in this dataset" for name, n in per_class.items() if n == 0],
        "verdict": "accepted",
    }


def read_detection_records(directory: str | Path) -> list[dict[str, Any]]:
    """Read BYOD records from ``<directory>/annotations.json`` plus the image and mask files it names.

    ``annotations.json`` is a list of ``{"file": "name.png", "boxes": [[x0, y0, x1, y1], ...], "labels": [...],
    "masks": ["name_mask0.png", ...]}`` objects; each mask file is a single-channel PNG whose non-zero pixels
    are the instance. File names must stay inside ``directory``; absolute paths and ``..`` are refused before
    any file is opened. The result still has to pass ``validate_dataset``.
    """
    root = Path(directory).resolve()
    index_path = root / "annotations.json"
    if not index_path.is_file():
        raise FileNotFoundError(f"{index_path} not found; expected annotations.json next to the images")
    with open(index_path, encoding="utf-8") as fh:
        entries = json.load(fh)
    if not isinstance(entries, list) or not entries:
        raise ValueError("annotations.json must hold a non-empty list of records")
    if len(entries) > MAX_RECORDS:
        raise ValueError(f"annotations.json lists {len(entries)} records > MAX_RECORDS {MAX_RECORDS}")

    def resolve(idx: int, name: Any) -> Path:
        relative = Path(str(name))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"annotations.json entry {idx}: file {name!r} must be relative to {root}")
        resolved = (root / relative).resolve()
        if root not in resolved.parents:
            raise ValueError(f"annotations.json entry {idx}: file {name!r} resolves outside {root}")
        return resolved

    records: list[dict[str, Any]] = []
    for idx, entry in enumerate(entries):
        if not isinstance(entry, dict) or not {"file", "boxes", "labels", "masks"} <= set(entry):
            raise ValueError(f"annotations.json entry {idx} must have 'file', 'boxes', 'labels' and 'masks'")
        image_path = resolve(idx, entry["file"])
        mask_paths = [resolve(idx, name) for name in entry["masks"]]
        with Image.open(image_path) as handle:
            image = handle.convert("RGB")
        masks = []
        for mask_path in mask_paths:
            with Image.open(mask_path) as handle:
                masks.append(np.asarray(handle.convert("L")) > 0)
        records.append({"id": str(entry["file"]), "image": image, "boxes": entry["boxes"], "labels": entry["labels"], "masks": masks})
    return records


def evaluation_report(
    result: Mapping[str, Any],
    references: Mapping[str, Mapping[str, Sequence[Any]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-image evaluation stage: per-object box_iou and mask_iou against same-label detections."""
    detections = list(result["detections"])
    masks = list(result.get("masks") or [])
    labels = tuple(result.get("class_names") or LABELS)
    base = {
        "task": f"instance segmentation over {len(labels)} class slots on one image",
        "decision_rule": (
            "an instance survives when its softmax class probability reaches the threshold after torchvision's "
            f"per-class non-maximum suppression; its mask is the pixels whose probability is at least {MASK_THRESHOLD}; "
            "neither value is a calibrated probability for the deployment's images"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "mask_threshold": MASK_THRESHOLD,
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not references:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference boxes or masks were supplied for the evaluated image",
            "needs": (
                "labelled instance masks per class on your own images, scored per object with box_iou and mask_iou "
                "and aggregated into box and mask average precision at stated IoU thresholds"
            ),
        }
    metrics = []
    for label, reference in references.items():
        if label not in labels:
            raise ValueError(f"unknown reference label {label!r}; expected one of {len(labels)} classes")
        same = [i for i, det in enumerate(detections) if det["label"] == label]
        for index, box in enumerate(reference["boxes"]):
            box_ious = [box_iou(detections[i]["box"], box) for i in same]
            best = same[int(np.argmax(box_ious))] if box_ious else None
            ref_mask = reference.get("masks", [None] * len(reference["boxes"]))[index]
            metrics.append(
                {
                    "id": "box_iou+mask_iou",
                    "reference": f"{label}-{index}",
                    "box_iou": max(box_ious) if box_ious else 0.0,
                    "mask_iou": mask_iou(masks[best], ref_mask) if best is not None and ref_mask is not None and masks else 0.0,
                    "matched_score": detections[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same),
                    "estimation": "one reference instance per object on a single image, no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": f"{len(metrics)} reference instance(s) on one tutorial image; geometry sanity evidence, not a benchmark",
        "needs": "a labelled image set from the deployment domain for any average-precision or precision/recall claim",
    }


def build_model(num_classes: int = len(COCO_CATEGORIES)) -> Any:
    """torchvision's maskrcnn_resnet50_fpn_v2 architecture with random weights and nothing downloaded."""
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2

    return maskrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=num_classes)


def _rehead(model: Any, num_labels: int) -> tuple[str, ...]:
    """Replace the box and mask predictors for ``num_labels`` classes plus background."""
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

    box_in = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_in, num_labels + 1)
    mask_in = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_in, 256, num_labels + 1)
    return ("roi_heads.box_predictor.", "roi_heads.mask_predictor.")


def _hold_batchnorm(module: Any) -> None:
    """Keep every BatchNorm layer under ``module`` in eval mode, so frozen layers keep their statistics."""
    import torch

    for sub in module.modules():
        if isinstance(sub, torch.nn.modules.batchnorm._BatchNorm):
            sub.eval()


@dataclass
class MaskRcnnPipeline:
    """COCO-class instance segmentation and transfer fine-tuning over Mask R-CNN ResNet-50 FPN v2."""

    model: Any
    device: str
    class_names: tuple[str, ...] = LABELS
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> MaskRcnnPipeline:
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no checkpoint manifest at {root}; stage {MODEL_ID} under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed file)"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        names = _check_class_names(class_names) if class_names is not None else LABELS

        import torch

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = build_model(len(COCO_CATEGORIES))
        state = torch.load(root / WEIGHTS_FILE, map_location="cpu", weights_only=True)
        model.load_state_dict(state, strict=True)
        reinitialised: tuple[str, ...] = ()
        if class_names is not None:
            torch.manual_seed(seed)
            reinitialised = _rehead(model, len(names))
        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            device=resolved_device,
            class_names=names,
            source=str(root),
            base_state_digest=MODEL_REVISION,
            reinitialised=reinitialised,
        )

    def _run(self, image: Image.Image, threshold: float) -> tuple[list[dict[str, Any]], list[np.ndarray]]:
        import torch
        from torchvision.transforms.functional import to_tensor

        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            output = self.model([to_tensor(image).to(self.device)])[0]
        if was_training:
            self.model.train()
        detections, masks = [], []
        for box, label, score, mask in zip(output["boxes"], output["labels"], output["scores"], output["masks"], strict=True):
            if float(score) < threshold:
                continue
            idx = int(label) - 1
            name = self.class_names[idx] if 0 <= idx < len(self.class_names) else f"class_{int(label)}"
            binary = (mask[0] >= MASK_THRESHOLD).cpu().numpy()
            detections.append({"box": [float(v) for v in box.tolist()], "label": name, "score": float(score), "mask_area": int(binary.sum())})
            masks.append(binary)
        return detections, masks

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Segment instances on one image; boxes are xyxy input pixels and masks are boolean (height, width)."""
        rgb, checked = _check_inputs(image, threshold)
        detections, masks = self._run(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(f"backend returned {len(detections)} detections > MAX_DETECTIONS {MAX_DETECTIONS}")
        for det, mask in zip(detections, masks, strict=True):
            if set(det) != {"box", "label", "score", "mask_area"} or len(det["box"]) != 4 or det["label"] not in self.class_names:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
            if mask.shape != (rgb.height, rgb.width):
                raise RuntimeError(f"backend returned a mask of shape {mask.shape} for a {rgb.size} image")
        order = sorted(range(len(detections)), key=lambda i: -detections[i]["score"])
        return {
            "detections": [detections[i] for i in order],
            "masks": [masks[i] for i in order],
            "threshold": checked,
            "mask_threshold": MASK_THRESHOLD,
            "width": rgb.width,
            "height": rgb.height,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        seed: int = DEFAULT_SEED,
        freeze_backbone: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning on ``records`` with torchvision's own Mask R-CNN losses.

        The loss is the sum torchvision returns in training mode: RPN objectness and box regression, ROI
        classification and box regression, and the per-pixel mask cross-entropy. With ``freeze_backbone`` the
        ResNet-50 body keeps its weights and its BatchNorm statistics (its BatchNorm layers are held in eval
        mode); the FPN, the RPN and the ROI heads are trained. Mutates this pipeline in place.
        """
        import torch
        from torchvision.transforms.functional import to_tensor

        validate_dataset(records, self.class_names, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if isinstance(learning_rate, bool) or not isinstance(learning_rate, int | float) or not 0.0 < float(learning_rate) <= 1.0:
            raise ValueError(f"learning_rate must be a number in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (freeze_backbone and name.startswith(BACKBONE_PREFIX))
        self.frozen_prefixes = (BACKBONE_PREFIX,) if freeze_backbone else ()
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(trainable, lr=float(learning_rate), momentum=DEFAULT_MOMENTUM, weight_decay=DEFAULT_WEIGHT_DECAY)
        class_to_id = {name: i + 1 for i, name in enumerate(self.class_names)}

        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            if freeze_backbone:
                _hold_batchnorm(self.model.backbone.body)
            order = rng.permutation(len(records))
            running, n_batches = 0.0, 0
            for start in range(0, len(records), batch_size):
                batch = [records[int(i)] for i in order[start : start + batch_size]]
                images = [to_tensor(validate_image(r["image"])).to(self.device) for r in batch]
                targets = [
                    {
                        "boxes": torch.tensor([[float(v) for v in b] for b in r["boxes"]], dtype=torch.float32, device=self.device),
                        "labels": torch.tensor([class_to_id[n] for n in r["labels"]], dtype=torch.int64, device=self.device),
                        "masks": torch.as_tensor(np.stack([np.asarray(m, dtype=np.uint8) for m in r["masks"]]), device=self.device),
                    }
                    for r in batch
                ]
                optimizer.zero_grad(set_to_none=True)
                losses = self.model(images, targets)
                loss = sum(losses.values())
                loss.backward()
                optimizer.step()
                running += float(loss.detach().cpu())
                n_batches += 1
            epoch_losses.append(running / max(1, n_batches))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})
        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "momentum": DEFAULT_MOMENTUM,
            "weight_decay": DEFAULT_WEIGHT_DECAY,
            "optimizer": "SGD",
            "seed": seed,
            "precision": "float32",
            "freeze_backbone": freeze_backbone,
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "loss": "torchvision Mask R-CNN training losses (RPN objectness + RPN box + ROI class + ROI box + mask)",
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_prefixes": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
        max_detections: int = MAX_EVAL_DETECTIONS,
    ) -> dict[str, Any]:
        """Score a labelled dataset: box and mask average precision plus evaluation metadata."""
        checked = _check_threshold(threshold, "threshold")
        predictions = []
        for record in records:
            result = self.detect(record["image"], threshold=checked)
            dets = [{**d, "mask": m} for d, m in zip(result["detections"], result["masks"], strict=True)][:max_detections]
            predictions.append(dets)
        box = average_precision(predictions, records, self.class_names, kind="box", iou_thresholds=iou_thresholds)
        mask = average_precision(predictions, records, self.class_names, kind="mask", iou_thresholds=iou_thresholds)
        return {
            "ap": box["ap"],
            "ap50": box["ap50"],
            "ap75": box["ap75"],
            "per_class_ap50": box["per_class_ap50"],
            "mask_ap": mask["ap"],
            "mask_ap50": mask["ap50"],
            "mask_ap75": mask["ap75"],
            "per_class_mask_ap50": mask["per_class_ap50"],
            "n_images": box["n_images"],
            "n_references": box["n_references"],
            "threshold": checked,
            "mask_threshold": MASK_THRESHOLD,
            "max_detections": max_detections,
            "adapted": self.adapted,
            "class_names": list(self.class_names),
            "estimation": f"one pass over {len(records)} held-out images; no resampling, no dispersion estimate",
            "implementation": (
                "package-local average_precision on boxes and on masks: greedy score-ordered matching and 101-point "
                "interpolation, without pycocotools area ranges or crowd handling"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base checkpoint, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base checkpoint.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "class_names": json.dumps(list(self.class_names)),
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}")
        return {**metadata, "class_names": json.loads(metadata["class_names"]), "frozen_prefixes": json.loads(metadata["frozen_prefixes"])}

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this (base) pipeline; refuse any tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        if tuple(metadata["class_names"]) != tuple(self.class_names):
            raise ValueError("artifact class_names differ from this pipeline's class_names")
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base checkpoint digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}")
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls,
        path: str | Path,
        *,
        weights_dir: str | Path | None = None,
        device: str | None = None,
    ) -> MaskRcnnPipeline:
        """Rebuild an adapted pipeline: verified base checkpoint first, then the adapter tensors."""
        metadata = cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, class_names=metadata["class_names"])
        pipe.apply_artifact(path)
        return pipe

**Module 2/2:** `src/maskrcnn_instance_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data with exact instance masks: the COCO demonstration scene and the
adaptation dataset.

Nothing here is downloaded and nothing needs torch — Pillow and numpy only. Every object is drawn twice:
once onto the RGB image and once onto its own single-channel mask, with the same geometry, so each
reference mask is exact by construction and each reference box is the tight box of that mask.

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns references drawn from the checkpoint's own **COCO** classes; it
  demonstrates the pretrained model and is not training data.
* ``sign_dataset`` returns records labelled with ``SIGN_CLASSES``, a three-class vocabulary that does
  **not** exist in COCO. It is the adaptation dataset, and a model fine-tuned on it answers in those
  three names only.
"""

from __future__ import annotations

import math
from collections.abc import Callable
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont

COCO_SCENE_SIZE = (640, 480)
ADAPT_SCENE_SIZE = (640, 640)

# The adaptation vocabulary. Deliberately not COCO names: "stop sign" exists in COCO, "yield-sign" and
# "speed-limit-sign" do not, and the hyphenated spellings keep the two vocabularies visually distinct.
SIGN_CLASSES: tuple[str, ...] = ("stop-sign", "yield-sign", "speed-limit-sign")


def _font(size: int) -> Any:
    return ImageFont.load_default(size=size)


def _mask_and_box(size: tuple[int, int], shape: Callable[[ImageDraw.ImageDraw], None]) -> tuple[np.ndarray, list[float]]:
    """Draw ``shape`` filled onto an empty mask; return the boolean mask and its tight xyxy box."""
    mask_image = Image.new("L", size, 0)
    shape(ImageDraw.Draw(mask_image))
    mask = np.asarray(mask_image) > 0
    ys, xs = np.nonzero(mask)
    if not len(xs):
        raise ValueError("shape drew nothing")
    return mask, [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def tutorial_scene(
    width: int = COCO_SCENE_SIZE[0], height: int = COCO_SCENE_SIZE[1]
) -> tuple[Image.Image, dict[str, dict[str, Any]]]:
    """The COCO demonstration scene: drawn objects with reference boxes and masks, keyed by COCO label.

    Returns ``(image, {label: {"boxes": [[x0, y0, x1, y1]], "masks": [bool HxW array]}})``. These are drawn
    references on a rendered picture, not a labelled photographic dataset: they support a per-object
    ``box_iou`` / ``mask_iou`` sanity check and nothing more.
    """
    size = (width, height)
    img = Image.new("RGB", size, (135, 190, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, int(height * 0.6875), width, height], fill=(96, 128, 72))
    d.rectangle([0, int(height * 0.625), width, int(height * 0.6875)], fill=(110, 110, 110))
    refs: dict[str, dict[str, Any]] = {}

    def add(label: str, shape: Callable[[ImageDraw.ImageDraw], None]) -> None:
        mask, box = _mask_and_box(size, shape)
        refs[label] = {"boxes": [box], "masks": [mask]}

    # Stop sign (the octagon only; the post is scenery)
    cx, cy, r = 110, 150, 62
    pts = [
        (cx + r * math.cos(math.pi / 8 + k * math.pi / 4), cy + r * math.sin(math.pi / 8 + k * math.pi / 4))
        for k in range(8)
    ]
    d.rectangle([cx - 5, cy, cx + 5, int(height * 0.6875)], fill=(90, 90, 90))
    d.polygon(pts, fill=(200, 20, 30), outline=(255, 255, 255))
    f = _font(30)
    d.text((cx - d.textlength("STOP", font=f) / 2, cy - 17), "STOP", fill="white", font=f)
    add("stop sign", lambda m, pts=pts: m.polygon(pts, fill=255, outline=255))

    # Traffic light (the housing)
    x0, y0 = 270, 60
    d.rectangle([x0 + 22, y0 + 150, x0 + 30, int(height * 0.6875)], fill=(70, 70, 70))
    d.rectangle([x0, y0, x0 + 52, y0 + 150], fill=(25, 25, 25), outline=(60, 60, 60))
    for k, col in enumerate([(230, 30, 30), (240, 200, 30), (40, 200, 60)]):
        d.ellipse([x0 + 8, y0 + 8 + k * 47, x0 + 44, y0 + 44 + k * 47], fill=col)
    add("traffic light", lambda m, x0=x0, y0=y0: m.rectangle([x0, y0, x0 + 52, y0 + 150], fill=255))

    # Clock (the face)
    cx, cy, r = 480, 140, 70
    d.rectangle([cx - 6, cy, cx + 6, int(height * 0.6875)], fill=(120, 80, 40))
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 250, 245), outline=(20, 20, 20), width=5)
    f2 = _font(16)
    for h in range(1, 13):
        a = math.radians(h * 30 - 90)
        d.text((cx + (r - 18) * math.cos(a) - 5, cy + (r - 18) * math.sin(a) - 8), str(h), fill="black", font=f2)
    d.line([(cx, cy), (cx + 0.5 * r * math.cos(math.radians(-60)), cy + 0.5 * r * math.sin(math.radians(-60)))], fill="black", width=5)
    d.line([(cx, cy), (cx + 0.8 * r * math.cos(math.radians(30)), cy + 0.8 * r * math.sin(math.radians(30)))], fill="black", width=3)
    add("clock", lambda m, cx=cx, cy=cy, r=r: m.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255))

    # Sports ball
    cx, cy, r = 330, 400, 45
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(235, 120, 30), outline=(40, 20, 10), width=3)
    d.line([(cx - r, cy), (cx + r, cy)], fill=(40, 20, 10), width=3)
    d.line([(cx, cy - r), (cx, cy + r)], fill=(40, 20, 10), width=3)
    d.arc([cx - r * 1.6, cy - r, cx - r * 0.2, cy + r], 300, 60, fill=(40, 20, 10), width=3)
    d.arc([cx + r * 0.2, cy - r, cx + r * 1.6, cy + r], 120, 240, fill=(40, 20, 10), width=3)
    add("sports ball", lambda m, cx=cx, cy=cy, r=r: m.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255))

    return img, refs


def blank_scene(width: int = 640, height: int = 640) -> Image.Image:
    """A featureless white image: the degenerate input every detector should be asked about."""
    return Image.new("RGB", (width, height), (255, 255, 255))


def noise_scene(seed: int = 0, width: int = 640, height: int = 640) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_scene``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (height, width, 3), dtype=np.uint8))


def _octagon_points(cx: float, cy: float, r: float) -> list[tuple[float, float]]:
    return [
        (cx + r * math.cos(math.pi / 8 + i * math.pi / 4), cy + r * math.sin(math.pi / 8 + i * math.pi / 4))
        for i in range(8)
    ]


def _draw_sign(
    draw: ImageDraw.ImageDraw, kind: str, cx: float, cy: float, r: float, limit: int
) -> Callable[[ImageDraw.ImageDraw], None]:
    """Draw one sign onto the RGB canvas; return the function that draws its filled silhouette on a mask."""
    if kind == "stop-sign":
        points = _octagon_points(cx, cy, r)
        draw.polygon(points, fill=(196, 30, 34), outline=(255, 255, 255))
        font = _font(int(max(11, r * 0.34)))
        draw.text((cx - draw.textlength("STOP", font=font) / 2, cy - r * 0.27), "STOP", fill=(255, 255, 255), font=font)
        return lambda m: m.polygon(points, fill=255, outline=255)
    if kind == "yield-sign":
        points = [(cx, cy + r), (cx - r * 0.95, cy - r * 0.75), (cx + r * 0.95, cy - r * 0.75)]
        draw.polygon(points, fill=(255, 255, 255), outline=(198, 32, 36))
        inner = [(cx, cy + r * 0.62), (cx - r * 0.62, cy - r * 0.5), (cx + r * 0.62, cy - r * 0.5)]
        draw.line([*inner, inner[0]], fill=(198, 32, 36), width=int(max(4, r * 0.22)))
        return lambda m: m.polygon(points, fill=255, outline=255)
    box = [cx - r, cy - r, cx + r, cy + r]
    draw.ellipse(box, fill=(255, 255, 255), outline=(198, 32, 36), width=int(max(4, r * 0.2)))
    font = _font(int(max(12, r * 0.9)))
    text = str(limit)
    draw.text((cx - draw.textlength(text, font=font) / 2, cy - r * 0.55), text, fill=(30, 30, 30), font=font)
    return lambda m: m.ellipse(box, fill=255)


def sign_dataset(
    n_images: int = 40,
    *,
    seed: int = 0,
    max_objects: int = 3,
    size: tuple[int, int] = ADAPT_SCENE_SIZE,
) -> list[dict[str, Any]]:
    """A deterministic labelled instance-segmentation dataset over ``SIGN_CLASSES`` for the bounded fine-tune.

    Each record is ``{"id": str, "image": PIL.Image, "boxes": [[x0, y0, x1, y1], ...], "labels": [name, ...],
    "masks": [bool HxW array, ...]}`` — the record shape ``validate_dataset`` and ``finetune`` accept, and
    the shape a BYOD caller must produce. Signs sit on a non-overlapping grid, so every mask is the sign's
    whole silhouette and every box is the tight box of its mask.

    It is synthetic drawn data, so a model fine-tuned on it learns to segment *these renderings*. That is
    the point of a bounded tutorial adaptation and the reason its metrics are not a claim about real
    traffic signs.
    """
    if not 1 <= n_images <= 500:
        raise ValueError(f"n_images must be in 1..500, got {n_images}")
    if not 1 <= max_objects <= 6:
        raise ValueError(f"max_objects must be in 1..6, got {max_objects}")
    rng = np.random.default_rng(seed)
    width, height = size
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records: list[dict[str, Any]] = []
    for index in range(n_images):
        image = Image.new("RGB", size, (232, 236, 240))
        draw = ImageDraw.Draw(image)
        tint = rng.integers(200, 245, 3)
        draw.rectangle([0, 0, width, height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0, int(height * 0.72), width, height], fill=(118, 122, 126))
        n_objects = int(rng.integers(1, max_objects + 1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes: list[list[float]] = []
        labels: list[str] = []
        masks: list[np.ndarray] = []
        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28, 29))
            cy += float(rng.integers(-28, 29))
            radius = float(rng.integers(46, 71))
            cx = min(max(cx, radius + 6), width - radius - 6)
            cy = min(max(cy, radius + 6), height - radius - 100)
            kind = SIGN_CLASSES[int(rng.integers(0, len(SIGN_CLASSES)))]
            limit = int(rng.choice([30, 50, 60, 80]))
            draw.rectangle([cx - 5, cy + radius * 0.6, cx + 5, min(height - 1, cy + radius + 90)], fill=(112, 112, 116))
            silhouette = _draw_sign(draw, kind, cx, cy, radius, limit)
            mask, box = _mask_and_box(size, silhouette)
            boxes.append(box)
            labels.append(kind)
            masks.append(mask)
        records.append(
            {"id": f"sign-{seed}-{index:03d}", "image": image, "boxes": boxes, "labels": labels, "masks": masks}
        )
    return records


def split_dataset(
    records: list[dict[str, Any]],
    *,
    train_fraction: float = 0.75,
    seed: int = 0,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Deterministically partition records into train and held-out splits (random, record-level).

    A random split assumes the records are independent. Records that share a source photograph, a
    scene or a camera session must be split by that group instead, or the held-out score leaks.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))
    n_train = min(len(records) - 1, max(1, int(len(records) * train_fraction)))
    train_idx = {int(i) for i in indices[:n_train]}
    train = [r for i, r in enumerate(records) if i in train_idx]
    held_out = [r for i, r in enumerate(records) if i not in train_idx]
    return train, held_out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the manifest URL on `download.pytorch.org` **and must hash to SHA-256 `unpinned…`**, `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_url_snapshot",
  "formatVersion": 1,
  "modelKey": "maskrcnn-resnet50-fpn-v2",
  "modelId": "torchvision/maskrcnn_resnet50_fpn_v2",
  "revision": "unpinned",
  "files": [
    {
      "path": "maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth",
      "url": "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth",
      "bytes": null,
      "sha256": null
    }
  ],
  "totalBytes": null
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained model does on a drawn scene

Before adapting anything, inspect the model you start from. The carried `samples` module draws a deterministic street scene with four objects, each with a reference box and an exact reference mask: a **stop sign**, a **traffic light**, an analogue **clock**, and an orange **sports ball**. All four are COCO classes.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant. Each score is the instance's **softmax class probability after torchvision's per-class non-maximum suppression (NMS), not a calibrated** probability for your images. The default `0.8` is the value torchvision's visualization-utilities gallery uses for this model, and it is passed explicitly on every call. A mask pixel belongs to the instance when its predicted probability is at least `MASK_THRESHOLD` (0.5).

The scene is drawn, not photographed, so a miss here is a finding about renderings, not about photographs. The evaluation report records every miss with `box_iou = 0.0` and `mask_iou = 0.0`. COCO mean average precision needs a labelled image set; on one scene the verdict is `sample-sanity`.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

threshold = 0.8  # @param {type:"number"}

print({'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS,
       'label_slots': len(LABELS), 'unannotated_slots': LABELS.count('N/A'), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD,
       'MASK_THRESHOLD': MASK_THRESHOLD})

scene, references = tutorial_scene()
buffer = io.BytesIO()
scene.save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(scene.size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'references': {label: len(ref['boxes']) for label, ref in references.items()}})

input_manifest = validate_inputs(scene, threshold=threshold, names=['tutorial-scene'])
try:
    validate_inputs(scene, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'threshold=1.5', 'rejected': str(exc)})
print({'verdict': input_manifest['verdict'], 'findings': input_manifest['findings'], 'inputs': input_manifest['inputs']})

coco_result = pipe.detect(scene, threshold=threshold)
for det in coco_result['detections']:
    print(f"{det['label']:>14s} {det['score']:.3f}  mask {det['mask_area']:>6d} px  [{', '.join(f'{v:.0f}' for v in det['box'])}]")

coco_report = evaluation_report(coco_result, references, sample_kind='synthetic')
print({'verdict': coco_report['verdict'], 'n_detections': coco_report['n_detections']})
for metric in coco_report['metrics']:
    print(f"  {metric['reference']:>18s}  box_iou {metric['box_iou']:.3f}  mask_iou {metric['mask_iou']:.3f}  same-label {metric['n_detected_same_label']}")
hits = sum(1 for m in coco_report['metrics'] if m['mask_iou'] >= 0.5)
print(f'{hits}/{len(coco_report["metrics"])} drawn objects matched at mask IoU >= 0.5')

overlay = np.asarray(scene).copy()
for mask in coco_result['masks']:
    overlay[mask] = (0.5 * overlay[mask] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
Image.fromarray(overlay)

## 5. Degenerate input probes: blank canvas and noise

Ask a segmentation model about structure-free input before trusting it. A model that returns confident instances on a blank canvas or on uniform noise will return them on empty real frames too. The cell counts instances on both images at the default threshold and at the evaluation threshold `EVAL_DETECTION_THRESHOLD` (0.05) that average precision is computed at.

**What to look for:** any instance above the default threshold on these two images is a false positive by construction.

In [ ]:
degenerate = {}
for name, image in (('blank', blank_scene()), ('noise', noise_scene(0))):
    standard = pipe.detect(image, threshold=threshold)['detections']
    lenient = pipe.detect(image, threshold=EVAL_DETECTION_THRESHOLD)['detections']
    degenerate[name] = {
        'at_default_threshold': len(standard),
        'at_evaluation_threshold': len(lenient),
        'top': [(d['label'], round(d['score'], 3), d['mask_area']) for d in lenient[:3]],
    }
print(json.dumps(degenerate, indent=2))

## 6. Labelled adaptation dataset and validation

Suppose your task needs sign classes that COCO does not have. `sign_dataset` draws a deterministic 40-image dataset over `SIGN_CLASSES`: `stop-sign`, `yield-sign` and `speed-limit-sign`. Every sign is drawn twice, once onto the image and once onto its own mask, so each reference mask is exact and each reference box is the tight box of its mask.

**Keep the two vocabularies apart.** COCO has `stop sign` (with a space). The adaptation vocabulary uses `stop-sign` (hyphenated), `yield-sign` and `speed-limit-sign`. They are different class identities, and the adapted model answers in the new names only.

`validate_dataset` checks every record before any model runs: the record keys, the image size ceilings, that every box is finite, non-empty and inside its image, that every mask has the image's shape, is boolean, is not empty and lies inside its box, and that every label is in the vocabulary. It returns a dataset manifest with the instance count per class and a finding for any class that has no instance.

In [ ]:
N_IMAGES = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 10  # @param {type:"integer"}

records = sign_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SIGN_CLASSES, epochs=EPOCHS)
print(json.dumps({k: v for k, v in dataset_manifest.items() if k != 'schema'}, indent=2))

preview = Image.new('RGB', (480, 320))
for index, record in enumerate(records[:3]):
    preview.paste(record['image'].resize((160, 160)), (160 * index, 0))
    union = np.any(np.stack(record['masks']), axis=0)
    preview.paste(Image.fromarray((union * 255).astype(np.uint8)).convert('RGB').resize((160, 160)), (160 * index, 160))
preview

## 7. Split, re-head, and measure the pre-adaptation baseline

The dataset is split at random into a training part (75%, 30 images) and a held-out part (25%, 10 images). A random split is valid here because every image is drawn independently; records that share a photograph or a camera session must be split by that group instead. The held-out images are never shown to the optimizer, and no hyperparameter is selected on them.

`from_pretrained(class_names=SIGN_CLASSES)` loads the verified checkpoint and replaces only the two class-specific predictors with freshly initialised layers: the box predictor (`roi_heads.box_predictor`, now 3 classes plus background) and the mask predictor (`roi_heads.mask_predictor`, one 28×28 mask channel per class). The backbone, the FPN, the RPN and the shared ROI layers keep their COCO weights.

**The baseline is expected to be near zero.** Randomly initialised predictors have no information about the new classes, so this number is the floor the fine-tune has to beat, not a property of Mask R-CNN.

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_dataset(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
overlap = {r['id'] for r in train_records} & {r['id'] for r in held_out}
assert not overlap, f'split leaked records: {sorted(overlap)}'
print({'train': len(train_records), 'held_out': len(held_out),
       'train_instances': sum(len(r['boxes']) for r in train_records),
       'held_out_instances': sum(len(r['boxes']) for r in held_out)})

adapter = MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SIGN_CLASSES, seed=SEED)
print({'class_names': list(adapter.class_names), 'device': adapter.device, 'reinitialised': list(adapter.reinitialised)})

baseline = adapter.evaluate(held_out)
print(json.dumps({key: round(baseline[key], 4) for key in ('ap', 'ap50', 'mask_ap', 'mask_ap50')} | {'n_references': baseline['n_references']}, indent=2))

## 8. Bounded fine-tuning

This cell runs the real adaptation step in this runtime. It is gradient fine-tuning with Mask R-CNN's own training losses, which torchvision returns when the model is called with targets: RPN objectness and RPN box regression, ROI classification and ROI box regression, and a per-pixel binary cross-entropy on the 28×28 mask of each positive region. The optimised loss is their sum.

- **The ResNet-50 body is frozen.** Parameters under `backbone.body.` keep their COCO values, and its BatchNorm layers are held in evaluation mode so their running statistics do not drift either. The FPN, the RPN and the ROI heads are trained. The cell prints the trainable and total parameter counts.
- **Schedule:** `EPOCHS` epochs of SGD at learning rate `0.005`, momentum 0.9 and weight decay 0.0005 (the values of torchvision's detection fine-tuning tutorial), batch size 2, float32, seed `SEED`.

**Read the loss as optimisation evidence only.** A falling loss says the optimizer is fitting the training images; the held-out average precision in the next section is the task evidence.

In [ ]:
LEARNING_RATE = 0.005  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_BACKBONE = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
print(json.dumps({key: run[key] for key in ('freeze_backbone', 'trainable_parameters', 'total_parameters', 'epochs',
                                         'batch_size', 'learning_rate', 'optimizer', 'precision', 'device')}, indent=2))

## 9. Evaluate on the held-out split

`evaluate` re-runs on the same held-out images with the same thresholds as the baseline, so the two rows are comparable. It scores every instance twice: once by box IoU (`ap`, `ap50`, `ap75`) and once by mask IoU (`mask_ap`, `mask_ap50`, `mask_ap75`). `ap50` is average precision at IoU 0.50; `ap` averages AP over the ten IoU thresholds 0.50–0.95, so it also rewards tight boxes and masks. A model can find every sign with a good box and still draw a poor mask, which only the mask rows show. These are tutorial metrics from one pass over 10 synthetic images, with no dispersion estimate.

In [ ]:
adapted = adapter.evaluate(held_out)
print(f"{'metric':<10s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('ap', 'ap50', 'ap75', 'mask_ap', 'mask_ap50', 'mask_ap75'):
    print(f"{key:<10s} {baseline[key]:>10.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}")
print('per-class mask AP50:', {k: round(v, 4) for k, v in adapted['per_class_mask_ap50'].items()})

## 10. Inference on unseen images

Three new images come from a seed the dataset never used (`NEW_DATA_SEED = 99`). The adapted pipeline segments the new sign classes, and each reference instance is compared with the best same-label instance by box IoU and by mask IoU.

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = sign_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for record in new_records:
    out = adapter.detect(record['image'], threshold=threshold)
    box_ious, mask_ious = [], []
    for box, label, mask in zip(record['boxes'], record['labels'], record['masks'], strict=True):
        same = [i for i, d in enumerate(out['detections']) if d['label'] == label]
        box_ious.append(round(max((box_iou(out['detections'][i]['box'], box) for i in same), default=0.0), 3))
        mask_ious.append(round(max((mask_iou(out['masks'][i], mask) for i in same), default=0.0), 3))
    row = {'id': record['id'], 'truth': record['labels'],
           'detections': [(d['label'], round(d['score'], 3)) for d in out['detections']],
           'same_label_box_iou': box_ious, 'same_label_mask_iou': mask_ious}
    new_data_rows.append(row)
    print(json.dumps(row))

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/maskrcnn_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned checkpoint SHA-256, the class names and the frozen prefixes. The frozen ResNet-50 body is left out because it equals the verified base checkpoint.

`load_artifact` then builds a **fresh** pipeline from the verified base checkpoint, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded instances with the in-memory model's on an unseen image, with a stated tolerance: loading succeeding is not the check, reproducing the boxes, scores and masks is.

In [ ]:
artifact_path = OUTPUTS / 'maskrcnn_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='Mask R-CNN ResNet-50 FPN v2 sign adaptation tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = MaskRcnnPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'class_names': list(reloaded.class_names)})

TOLERANCE = 1e-3
MASK_AGREEMENT = 0.99
test_img = new_records[0]['image']
out_orig = adapter.detect(test_img, threshold=EVAL_DETECTION_THRESHOLD)
out_reloaded = reloaded.detect(test_img, threshold=EVAL_DETECTION_THRESHOLD)
assert len(out_orig['detections']) == len(out_reloaded['detections'])
for d1, d2, m1, m2 in zip(out_orig['detections'], out_reloaded['detections'], out_orig['masks'], out_reloaded['masks'], strict=True):
    assert d1['label'] == d2['label']
    assert np.allclose(d1['box'], d2['box'], atol=TOLERANCE)
    assert abs(d1['score'] - d2['score']) <= TOLERANCE
    assert (not m1.any() and not m2.any()) or mask_iou(m1, m2) >= MASK_AGREEMENT
reload_check = {'instances_compared': len(out_orig['detections']), 'tolerance': TOLERANCE, 'mask_iou_at_least': MASK_AGREEMENT, 'equivalent': True}
print(reload_check)

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/maskrcnn_instance_segmentation_input_manifest.json`
- `outputs/maskrcnn_instance_segmentation_evaluation_report.json`
- `outputs/maskrcnn_instance_segmentation_result.json` (identity, runtime versions, device, dataset manifest, split, baseline and adapted metrics, fine-tuning configuration, new-data rows, adapter descriptor and reload check)
- `outputs/maskrcnn_instance_segmentation_detections.csv` (one row per instance, with its mask area)
- `outputs/maskrcnn_instance_segmentation_masks.npz` (the demonstration scene's boolean masks)
- `outputs/maskrcnn_instance_segmentation_annotated.png`
- `outputs/maskrcnn_adapter.safetensors` (written in Section 11)

In [ ]:
import csv
from PIL import ImageDraw

with open(OUTPUTS / 'maskrcnn_instance_segmentation_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'maskrcnn_instance_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(coco_report, f, indent=2)

np.savez_compressed(OUTPUTS / 'maskrcnn_instance_segmentation_masks.npz', *coco_result['masks'])

annotated_pixels = np.asarray(scene).copy()
for mask in coco_result['masks']:
    annotated_pixels[mask] = (0.5 * annotated_pixels[mask] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
annotated = Image.fromarray(annotated_pixels)
draw = ImageDraw.Draw(annotated)
for det in coco_result['detections']:
    x0, y0, x1, y1 = det['box']
    draw.rectangle([x0, y0, x1, y1], outline='red', width=3)
    draw.text((x0 + 4, y0 + 4), f"{det['label']} {det['score']:.2f}", fill='red')
annotated.save(OUTPUTS / 'maskrcnn_instance_segmentation_annotated.png')

with open(OUTPUTS / 'maskrcnn_instance_segmentation_detections.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1', 'mask_area'])
    for rank, d in enumerate(coco_result['detections']):
        writer.writerow(['tutorial-scene', rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box']), d['mask_area']])
    for row, record in zip(new_data_rows, new_records, strict=True):
        for rank, d in enumerate(adapter.detect(record['image'], threshold=threshold)['detections']):
            writer.writerow([row['id'], rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box']), d['mask_area']])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'weights_url': WEIGHTS_URL,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'threshold': threshold,
    'mask_threshold': MASK_THRESHOLD,
    'coco_detections': coco_result['detections'],
    'degenerate_probes': degenerate,
    'adaptation': {
        'dataset': {k: v for k, v in dataset_manifest.items() if k != 'schema'},
        'dataset_seed': DATASET_SEED,
        'split': {'train': len(train_records), 'held_out': len(held_out), 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': run,
        'baseline': baseline,
        'adapted': adapted,
        'new_data': new_data_rows,
        'artifact': descriptor,
        'reload_check': reload_check,
    },
}
with open(OUTPUTS / 'maskrcnn_instance_segmentation_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<44s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Image branch** (`USE_BYOD_IMAGE`): one image file PIL can open, each side between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px. It runs through `validate_inputs`, `detect` and `evaluation_report` — the report is `not-measurable`, because no reference instances come with it.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory with `annotations.json` (a list of `{'file', 'boxes', 'labels', 'masks'}` objects: xyxy pixel boxes and one mask PNG name per instance) and the image and mask files it names, at least 2 records and at most 5,000. File names must stay inside the directory. `BYOD_CLASS_NAMES` is a comma-separated vocabulary; leave it empty to use the sorted set of labels in the annotations. The branch runs the same validate → split → baseline → fine-tune → evaluate → export → reload stages as the sample.

Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_DIR` to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead. Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. The first lines of the cell show the validator refusing two malformed inputs with messages that name the failed rule.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_DIR = ""  # @param {type:"string"}
BYOD_CLASS_NAMES = ""  # @param {type:"string"}

_probe_image = blank_scene()
for desc, probe in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png')),
    ('empty mask', lambda: validate_dataset([{'image': _probe_image, 'boxes': [[0, 0, 10, 10]], 'labels': [SIGN_CLASSES[0]],
                                               'masks': [np.zeros((_probe_image.height, _probe_image.width), dtype=bool)]}], SIGN_CLASSES)),
):
    try:
        probe()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

BYOD_DIR = OUTPUTS / 'byod'

def _upload_into(target):
    from google.colab import files  # type: ignore[import-not-found]
    target.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (target / Path(name).name).write_bytes(data)
    return target

if USE_BYOD_IMAGE:
    image_path = Path(BYOD_IMAGE_PATH) if BYOD_IMAGE_PATH else next(iter(sorted(_upload_into(BYOD_DIR / 'image').iterdir())))
    with Image.open(image_path) as handle:
        byod_image = handle.convert('RGB')
    print(validate_inputs(byod_image, threshold=threshold, names=[image_path.name])['verdict'])
    byod_result = pipe.detect(byod_image, threshold=threshold)
    for det in byod_result['detections'][:20]:
        print(f"{det['label']:>16s} {det['score']:.3f}  mask {det['mask_area']:>7d} px  [{', '.join(f'{v:.0f}' for v in det['box'])}]")
    print(evaluation_report(byod_result, None, sample_kind='byod')['verdict'])
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True to run segmentation on your own image.')

if USE_BYOD_DATASET:
    dataset_dir = Path(BYOD_DATASET_DIR) if BYOD_DATASET_DIR else _upload_into(BYOD_DIR / 'dataset')
    byod_records = read_detection_records(dataset_dir)
    byod_names = [n.strip() for n in BYOD_CLASS_NAMES.split(',') if n.strip()] or sorted({l for r in byod_records for l in r['labels']})
    byod_manifest = validate_dataset(byod_records, byod_names, epochs=EPOCHS)
    print(json.dumps({k: v for k, v in byod_manifest.items() if k != 'schema'}, indent=2))
    byod_train, byod_held = split_dataset(byod_records, train_fraction=1.0 - HOLDOUT, seed=SEED)
    byod_pipe = MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=byod_names, seed=SEED)
    byod_baseline = byod_pipe.evaluate(byod_held)
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_backbone=FREEZE_BACKBONE)
    byod_adapted = byod_pipe.evaluate(byod_held)
    print({key: (round(byod_baseline[key], 4), round(byod_adapted[key], 4)) for key in ('ap50', 'mask_ap50')})
    byod_descriptor = byod_pipe.save_artifact(OUTPUTS / 'byod_maskrcnn_adapter.safetensors', notes='BYOD adaptation adapter')
    MaskRcnnPipeline.load_artifact(OUTPUTS / 'byod_maskrcnn_adapter.safetensors', weights_dir=WEIGHTS_DIR)
    print('BYOD adapter exported and reloaded:', byod_descriptor['sha256'][:16])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt Mask R-CNN on your own labelled images.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned torchvision checkpoint for `torchvision/maskrcnn_resnet50_fpn_v2` was verified against a committed SHA-256 manifest before loading. The pretrained model was run on a drawn scene and on two structure-free probes, and its boxes and masks were compared with the drawn references by IoU. A three-class sign vocabulary that COCO does not contain was then adapted by replacing the box and mask predictors, training the FPN, RPN and ROI heads with the ResNet-50 body frozen, and scoring the held-out split with box and mask average precision before and after. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory instances.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the synthetic images it was fitted to. The held-out AP is measured on 10 drawn images and carries no dispersion estimate. The scores are not calibrated probabilities.

**Reproducibility.** Seeds are form fields (`DATASET_SEED`, `SEED`, `NEW_DATA_SEED`), the run is float32 with no data augmentation, and the new predictors are initialised under `SEED`. GPU kernels (ROI Align in particular) are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**Try next.** Change `FREEZE_BACKBONE` to `False` and compare held-out mask AP and runtime, or change `EPOCHS` and watch where held-out mask AP stops improving. To transfer the workflow, point `BYOD_DATASET_DIR` at a small labelled set from your own domain.

## References

- He, K., Gkioxari, G., Dollár, P. and Girshick, R. (2017). *Mask R-CNN.* [arXiv:1703.06870](https://arxiv.org/abs/1703.06870).
- Li, Y., Xie, S., Chen, X., Dollár, P., He, K. and Girshick, R. (2021). *Benchmarking Detection Transfer Learning with Vision Transformers.* [arXiv:2111.11429](https://arxiv.org/abs/2111.11429) — the recipe behind the v2 weights.
- torchvision model documentation: [maskrcnn_resnet50_fpn_v2](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html); upstream repository [pytorch/vision](https://github.com/pytorch/vision) — BSD-3-Clause.
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* [arXiv:1405.0312](https://arxiv.org/abs/1405.0312).
- Repository model card: https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/maskrcnn-instance-segmentation-pipeline`](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline) — source repository for this pipeline.